In [ ]:
# =============================================================================
# MANTIS v31.0 – SOFT PROBABILITY ENSEMBLE (RANK-BASED WEIGHTING FIX)
# Fixed: Weights now properly differentiate between agents
# =============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.signal import butter, filtfilt
from openai import OpenAI
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# CONFIGURATION
# -------------------------------
SEQ_LEN = 30
STRIDE = 1
BATCH_SIZE = 64
MAX_RUL = 125
EPOCHS = 120
PATIENCE = 15
LR_INIT = 0.001
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-4
WEIGHT_TEMPERATURE = 2.0      # Controls rank score spread
WEIGHT_MOMENTUM = 0.9         # Smooth weight updates
NVIDIA_API_KEY = "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"

# -------------------------------
# Check GPU
# -------------------------------
if not torch.cuda.is_available():
    print("Warning: CUDA not available, using CPU.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------------------
# 1. LOW‑PASS FILTER (Butterworth)
# -------------------------------
def butter_lowpass(cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def lowpass_filter(data, cutoff=0.1, fs=1.0, order=4):
    b, a = butter_lowpass(cutoff, fs, order=order)
    return filtfilt(b, a, data)

def apply_lowpass_per_engine(df, sensor_cols, cutoff=0.1, fs=1.0):
    df_filtered = df.copy()
    for engine_id in df['engine_id'].unique():
        mask = df['engine_id'] == engine_id
        for col in sensor_cols:
            raw = df.loc[mask, col].values
            if len(raw) >= 10:
                filtered = lowpass_filter(raw, cutoff, fs)
            else:
                filtered = raw
            df_filtered.loc[mask, col] = filtered
    return df_filtered

# -------------------------------
# 2. DATA LOADING (StandardScaler + Low‑Pass Filter + Trends)
# -------------------------------
def load_cmapss_with_denoising(path_train, path_test, path_rul):
    cols = ["engine_id", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
    train = pd.read_csv(path_train, sep=r"\s+", header=None, names=cols)
    test = pd.read_csv(path_test, sep=r"\s+", header=None, names=cols)
    rul = pd.read_csv(path_rul, sep=r"\s+", header=None, names=["rul"])

    # RUL calculation
    max_cycle = train.groupby("engine_id")["cycle"].max().reset_index()
    max_cycle.columns = ["engine_id", "max_cycle"]
    train = train.merge(max_cycle, on="engine_id")
    train["RUL"] = (train["max_cycle"] - train["cycle"]).clip(upper=MAX_RUL)
    train.drop("max_cycle", axis=1, inplace=True)

    test_max = test.groupby("engine_id")["cycle"].max().reset_index()
    test_max.columns = ["engine_id", "max_cycle"]
    test = test.merge(test_max, on="engine_id")
    test["RUL"] = test["max_cycle"] - test["cycle"]
    test["true_RUL"] = test["engine_id"].map({i+1: rul.iloc[i, 0] for i in range(len(rul))})
    test["RUL"] = (test["true_RUL"] + test["RUL"]).clip(upper=MAX_RUL)
    test.drop(["max_cycle", "true_RUL"], axis=1, inplace=True)

    # Step 1: StandardScaler on raw sensors
    sensor_cols = [f"s{i}" for i in range(1, 22)]
    scaler_sensors = StandardScaler()
    train[sensor_cols] = scaler_sensors.fit_transform(train[sensor_cols])
    test[sensor_cols] = scaler_sensors.transform(test[sensor_cols])

    # Step 2: Low‑pass filter per engine
    train = apply_lowpass_per_engine(train, sensor_cols, cutoff=0.1, fs=1.0)
    test = apply_lowpass_per_engine(test, sensor_cols, cutoff=0.1, fs=1.0)

    # Step 3: Trend features (rolling mean + difference)
    trend_sensors = [f"s{i}" for i in [2, 3, 4, 7, 11, 12, 15, 17, 20, 21]]
    window_size = 10
    def add_trend_features(df, sensor_list, win):
        for sn in sensor_list:
            df[f"{sn}_mean"] = df.groupby("engine_id")[sn].transform(
                lambda x: x.rolling(window=win, min_periods=1).mean()
            )
            df[f"{sn}_trend"] = df[sn] - df[f"{sn}_mean"]
        return df
    train = add_trend_features(train, trend_sensors, window_size)
    test = add_trend_features(test, trend_sensors, window_size)

    # Step 4: Final scaling
    ops_cols = ['op1','op2','op3']
    trend_cols = [f"{s}_mean" for s in trend_sensors] + [f"{s}_trend" for s in trend_sensors]
    all_feat = ops_cols + sensor_cols + trend_cols
    scaler_final = RobustScaler()
    train[all_feat] = scaler_final.fit_transform(train[all_feat])
    test[all_feat] = scaler_final.transform(test[all_feat])

    return train, test, all_feat

# -------------------------------
# 3. FIVE AGENTS
# -------------------------------
class LSTMAgent(nn.Module):
    def __init__(self, input_size=44, hidden=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.drop(out)
        return self.fc(out).squeeze(-1)

class VAEAntiDenoise(nn.Module):
    def __init__(self, input_size=44, latent_dim=16, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.input_dim = input_size * seq_len
        self.enc_fc1 = nn.Linear(self.input_dim, 128)
        self.enc_fc2 = nn.Linear(128, 64)
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, 64)
        self.dec_fc2 = nn.Linear(64, 128)
        self.dec_out = nn.Linear(128, self.input_dim)
        self.reg = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 1))
    def encode(self, x):
        x = x.view(x.size(0), -1)
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-5, max=5)
        std = torch.exp(0.5 * logvar) + 1e-6
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self, x_seq):
        mu, logvar = self.encode(x_seq)
        z = self.reparameterize(mu, logvar)
        rul = self.reg(z).squeeze(-1)
        x_flat = x_seq.view(x_seq.size(0), -1)
        return rul, mu, logvar, x_flat

class TransformerAgent(nn.Module):
    def __init__(self, input_size=44, window_size=30, d_model=64, nhead=4, num_layers=3):
        super().__init__()
        self.window_size = window_size
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=0.3, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.pos_embed = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
    def forward(self, x):
        if x.size(1) < self.window_size:
            pad = self.window_size - x.size(1)
            x = F.pad(x, (0,0,0,pad))
        x = x[:, -self.window_size:, :]
        x = self.input_proj(x)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.transformer(x)
        out = x[:, -1, :]
        out = torch.nan_to_num(out, nan=0.0)
        return self.fc(out).squeeze(-1)

class CNN1DAgent(nn.Module):
    def __init__(self, input_size=44, channels=[32, 64], kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, channels[0], kernel_size, padding=1)
        self.conv2 = nn.Conv1d(channels[0], channels[1], kernel_size, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(channels[1], 1)
        self.drop = nn.Dropout(0.4)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        x = self.drop(x)
        return self.fc(x).squeeze(-1)

# -------------------------------
# 4. GENAI META-OPTIMIZER
# -------------------------------
class GenAIMetaOptimizer:
    def __init__(self, api_key, memory_length=3, min_lr=1e-4):
        self.client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)
        self.memory_length = memory_length
        self.history = []
        self.min_lr = min_lr

    def diagnose(self, epoch, val_rmses, weights, lrs, ensemble_rmse):
        context = ""
        if self.history:
            context = "Previous checks:\n"
            for rec in self.history[-self.memory_length:]:
                context += f"Epoch {rec['epoch']}: RMSEs={rec['val_rmses']}, weights={rec['weights']}, action='{rec['action']}'\n"
        agent_names = ["LSTM", "VAE", "Transformer-L", "Transformer-S", "CNN"]
        prompt = f"""
MANTIS v31.0 ensemble (rank-based weighting). Epoch {epoch}.
Val RMSE per agent: {dict(zip(agent_names, [round(x,3) for x in val_rmses]))}
Ensemble weights: {[round(w,2) for w in weights]}
LRs: {lrs}
Ensemble RMSE: {ensemble_rmse:.3f}
{context}
Suggest ONE action: "reduce LR of [AgentName]" only if LR > {self.min_lr}, else "keep current".
Output only the action.
"""
        try:
            resp = self.client.chat.completions.create(
                model="nvidia/nemotron-3-super-120b-a12b",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=0.2
            )
            action = resp.choices[0].message.content.strip()
            print(f"[GenAI] → {action}")
            self.history.append({'epoch': epoch, 'val_rmses': val_rmses.copy(),
                                 'weights': weights.copy(), 'action': action})
            return action
        except Exception as e:
            print(f"[GenAI] Error: {e}")
            return "keep current"

# -------------------------------
# 5. TRAINING LOOP WITH RANK-BASED WEIGHTING (FIXED)
# -------------------------------
def train_ensemble(agents, train_loader, val_loader, val_labels, epochs, patience, genai=None):
    # Per-agent learning rates (lower for unstable agents)
    optimizers = [
        torch.optim.AdamW(agents[0].parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY),      # LSTM
        torch.optim.AdamW(agents[1].parameters(), lr=LR_INIT*0.3, weight_decay=WEIGHT_DECAY),  # VAE
        torch.optim.AdamW(agents[2].parameters(), lr=LR_INIT*0.3, weight_decay=WEIGHT_DECAY),  # Transformer-L
        torch.optim.AdamW(agents[3].parameters(), lr=LR_INIT*0.3, weight_decay=WEIGHT_DECAY),  # Transformer-S
        torch.optim.AdamW(agents[4].parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)       # CNN
    ]
    schedulers = [torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.8, patience=5, min_lr=LR_MIN)
                  for opt in optimizers]

    best_val_rmse = float('inf')
    wait = 0
    best_weights = None
    weights = np.ones(len(agents)) / len(agents)
    prev_weights = weights.copy()

    for epoch in range(epochs):
        # Training
        for idx, agent in enumerate(agents):
            agent.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                optimizers[idx].zero_grad()
                if isinstance(agent, VAEAntiDenoise):
                    rul, mu, logvar, x_flat = agent(Xb)
                    rul_loss = F.huber_loss(rul, yb, delta=0.5)
                    h = F.relu(agent.dec_fc1(mu))
                    h = F.relu(agent.dec_fc2(h))
                    recon = agent.dec_out(h)
                    recon_loss = F.mse_loss(recon, x_flat)
                    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / Xb.size(0)
                    loss = rul_loss + 0.001 * recon_loss + 0.001 * kl_loss
                else:
                    pred = agent(Xb)
                    loss = F.huber_loss(pred, yb, delta=0.5)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(agent.parameters(), 1.0)
                optimizers[idx].step()

        # Validation
        all_preds = []
        for idx, agent in enumerate(agents):
            agent.eval()
            preds = []
            with torch.no_grad():
                for Xb, _ in val_loader:
                    Xb = Xb.to(DEVICE)
                    if isinstance(agent, VAEAntiDenoise):
                        rul, _, _, _ = agent(Xb)
                        preds.extend(rul.cpu().numpy())
                    else:
                        preds.extend(agent(Xb).cpu().numpy())
            preds_arr = np.array(preds)
            if np.isnan(preds_arr).any():
                print(f"Warning: NaN in agent {idx} predictions. Replacing with 0.")
                preds_arr = np.nan_to_num(preds_arr, nan=0.0)
            all_preds.append(preds_arr)

        val_rmses = [np.sqrt(mean_squared_error(val_labels, p)) for p in all_preds]
        val_rmses_arr = np.array(val_rmses)

        # ========== RANK-BASED WEIGHTING (FIXED) ==========
        # Convert RMSE to ranks (lower RMSE = higher rank)
        ranks = np.argsort(np.argsort(val_rmses_arr))  # 0 = best, 4 = worst
        rank_scores = len(ranks) - ranks  # higher score for better agents (5 to 1)
        
        # Exponential weighting based on rank
        raw_weights = np.exp(rank_scores / WEIGHT_TEMPERATURE)
        raw_weights = raw_weights / raw_weights.sum()
        
        # Print debug info every 5 epochs
        if epoch % 5 == 0 or epoch == 0:
            agent_names = ["LSTM", "VAE", "Tr-L", "Tr-S", "CNN"]
            print(f"  Agent RMSEs: {dict(zip(agent_names, [round(x,2) for x in val_rmses_arr]))}")
            print(f"  Rank scores: {rank_scores}")
            print(f"  Raw weights: {raw_weights.round(3)}")
        
        # Apply momentum smoothing
        if epoch == 0:
            weights = raw_weights.copy()
        else:
            weights = WEIGHT_MOMENTUM * prev_weights + (1 - WEIGHT_MOMENTUM) * raw_weights
            weights = weights / weights.sum()
        prev_weights = weights.copy()
        # ==================================================

        # Ensemble prediction
        ensemble_pred = np.zeros_like(all_preds[0])
        for i, p in enumerate(all_preds):
            ensemble_pred += weights[i] * np.array(p)
        ensemble_rmse = np.sqrt(mean_squared_error(val_labels, ensemble_pred))

        # Step schedulers
        for idx, sch in enumerate(schedulers):
            sch.step(val_rmses[idx])

        # Print epoch summary
        current_lrs = [opt.param_groups[0]['lr'] for opt in optimizers]
        print(f"Epoch {epoch:3d} | Ens RMSE: {ensemble_rmse:.4f} | Weights: {weights.round(3)} | LRs: {[round(lr,5) for lr in current_lrs]}")

        # GenAI intervention (every 10 epochs)
        if genai and epoch % 10 == 0 and epoch > 0:
            action = genai.diagnose(epoch, val_rmses, weights, current_lrs, ensemble_rmse)
            action_lower = action.lower()
            agent_names_lower = ["lstm", "vae", "transformer-l", "transformer-s", "cnn"]
            for i, name in enumerate(agent_names_lower):
                if name in action_lower and "reduce lr" in action_lower:
                    old_lr = optimizers[i].param_groups[0]['lr']
                    if old_lr > LR_MIN:
                        new_lr = max(old_lr * 0.7, LR_MIN)
                        optimizers[i].param_groups[0]['lr'] = new_lr
                        print(f"  → LR of {agents[i].__class__.__name__}: {old_lr:.2e} -> {new_lr:.2e}")

        # Early stopping
        if ensemble_rmse < best_val_rmse:
            best_val_rmse = ensemble_rmse
            best_weights = weights.copy()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_weights, best_val_rmse

# -------------------------------
# 6. SEQUENCE CREATION
# -------------------------------
def create_sequences_full(df, seq_len, feature_cols, return_engine_ids=False):
    X, y, eng_ids = [], [], []
    for eid in df['engine_id'].unique():
        eng = df[df['engine_id'] == eid].sort_values('cycle')
        f = eng[feature_cols].values
        t = eng['RUL'].values
        for i in range(len(eng) - seq_len):
            X.append(f[i:i+seq_len])
            y.append(t[i+seq_len])
            eng_ids.append(eid)
    if return_engine_ids:
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(eng_ids)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# -------------------------------
# 7. MAIN
# -------------------------------
def main(dataset="FD001"):
    # Dataset paths
    if dataset == "FD001":
        train_path = "/kaggle/input/datasets/jameskariukimwaura/fd001-1/train_FD001.txt"
        test_path  = "/kaggle/input/datasets/jameskariukimwaura/fd001-1/test_FD001.txt"
        rul_path   = "/kaggle/input/datasets/jameskariukimwaura/fd001-1/RUL_FD001.txt"
        baseline = 14.2
    elif dataset == "FD002":
        base = "/kaggle/input/datasets/jameskariukimwaura/fd002"
        train_path = f"{base}/train_FD002.txt"
        test_path  = f"{base}/test_FD002.txt"
        rul_path   = f"{base}/RUL_FD002.txt"
        baseline = 22.0
    elif dataset == "FD003":
        base = "/kaggle/input/datasets/jameskariukimwaura/fd003"
        train_path = f"{base}/train_FD003.txt"
        test_path  = f"{base}/test_FD003.txt"
        rul_path   = f"{base}/RUL_FD003.txt"
        baseline = 14.2
    elif dataset == "FD004":
        base = "/kaggle/input/datasets/jameskariukimwaura/fd004"
        train_path = f"{base}/train_FD004.txt"
        test_path  = f"{base}/test_FD004.txt"
        rul_path   = f"{base}/RUL_FD004.txt"
        baseline = 22.0
    else:
        raise ValueError("Dataset not recognized")

    print("="*80)
    print(f"MANTIS v31.0 – RANK-BASED WEIGHTING on {dataset}")
    print("="*80)

    train_df, test_df, feature_cols = load_cmapss_with_denoising(train_path, test_path, rul_path)

    X_train_full, y_train_full, train_eng_ids = create_sequences_full(
        train_df, SEQ_LEN, feature_cols, return_engine_ids=True
    )
    X_test, y_test = create_sequences_full(test_df, SEQ_LEN, feature_cols, return_engine_ids=False)

    # Engine-wise validation split (last 20% of engines)
    unique_engines = np.unique(train_eng_ids)
    n_val_engines = max(1, int(0.2 * len(unique_engines)))
    val_engines = unique_engines[-n_val_engines:]
    train_engines = unique_engines[:-n_val_engines]

    train_mask = np.isin(train_eng_ids, train_engines)
    val_mask = np.isin(train_eng_ids, val_engines)

    X_tr, y_tr = X_train_full[train_mask], y_train_full[train_mask]
    X_val, y_val = X_train_full[val_mask], y_train_full[val_mask]

    print(f"Train: {X_tr.shape} from {len(train_engines)} engines")
    print(f"Val:   {X_val.shape} from {len(val_engines)} engines")
    print(f"Test:  {X_test.shape}")

    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).float()),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).float()),
                            batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float()),
                             batch_size=BATCH_SIZE, shuffle=False)

    # Initialize five agents
    agents = [
        LSTMAgent(input_size=len(feature_cols), hidden=64, num_layers=2).to(DEVICE),
        VAEAntiDenoise(input_size=len(feature_cols), latent_dim=16, seq_len=SEQ_LEN).to(DEVICE),
        TransformerAgent(input_size=len(feature_cols), window_size=50, d_model=64, nhead=4, num_layers=3).to(DEVICE),
        TransformerAgent(input_size=len(feature_cols), window_size=15, d_model=64, nhead=4, num_layers=2).to(DEVICE),
        CNN1DAgent(input_size=len(feature_cols), channels=[32,64], kernel_size=3).to(DEVICE)
    ]
    print("\nFive agents initialised (LSTM, VAE, Transformer-L, Transformer-S, CNN).\n")

    genai = GenAIMetaOptimizer(NVIDIA_API_KEY, memory_length=3, min_lr=LR_MIN)

    best_weights, best_val = train_ensemble(agents, train_loader, val_loader, y_val,
                                            EPOCHS, PATIENCE, genai=genai)

    # Test evaluation
    all_preds = []
    for agent in agents:
        agent.eval()
        preds = []
        with torch.no_grad():
            for Xb, _ in test_loader:
                Xb = Xb.to(DEVICE)
                if isinstance(agent, VAEAntiDenoise):
                    rul, _, _, _ = agent(Xb)
                    preds.extend(rul.cpu().numpy())
                else:
                    preds.extend(agent(Xb).cpu().numpy())
        all_preds.append(preds)

    final_pred = np.zeros_like(all_preds[0])
    for i, p in enumerate(all_preds):
        final_pred += best_weights[i] * np.array(p)
    final_pred = np.clip(final_pred, 0, MAX_RUL)

    rmse = np.sqrt(mean_squared_error(y_test, final_pred))
    mae = mean_absolute_error(y_test, final_pred)
    r2 = r2_score(y_test, final_pred)

    print("\n" + "="*60)
    print(f"FINAL RESULTS - MANTIS v31.0 on {dataset}")
    print(f"Test RMSE        : {rmse:.4f} cycles")
    print(f"Test MAE         : {mae:.4f} cycles")
    print(f"Test R²          : {r2:.4f}")
    print(f"Baseline         : {baseline:.2f} cycles")
    print(f"Improvement      : {(baseline - rmse)/baseline*100:+.1f}%")
    print("="*60)

    plt.figure(figsize=(8,6))
    plt.scatter(y_test, final_pred, alpha=0.3, s=5)
    plt.plot([0,MAX_RUL], [0,MAX_RUL], 'r--')
    plt.xlabel('True RUL')
    plt.ylabel('Predicted RUL')
    plt.title(f'MANTIS v31.0 on {dataset} – RMSE = {rmse:.2f}')
    plt.savefig(f"mantis_v31_{dataset}.png")
    print(f"Plot saved as 'mantis_v31_{dataset}.png'")

if __name__ == "__main__":
    # Change dataset: "FD001", "FD002", "FD003", "FD004"
    main(dataset="FD001")

Using device: cuda
MANTIS v31.0 – RANK-BASED WEIGHTING on FD001
Train: (13738, 30, 44) from 80 engines
Val:   (3893, 30, 44) from 20 engines
Test:  (10096, 30, 44)

Five agents initialised (LSTM, VAE, Transformer-L, Transformer-S, CNN).

  Agent RMSEs: {'LSTM': np.float64(78.77), 'VAE': np.float64(94.38), 'Tr-L': np.float64(94.38), 'Tr-S': np.float64(94.38), 'CNN': np.float64(73.08)}
  Rank scores: [4 2 3 1 5]
  Raw weights: [0.26  0.096 0.158 0.058 0.429]
Epoch   0 | Ens RMSE: 78.8247 | Weights: [0.26  0.096 0.158 0.058 0.429] | LRs: [0.001, 0.0003, 0.0003, 0.0003, 0.001]
Epoch   1 | Ens RMSE: 73.8123 | Weights: [0.277 0.096 0.158 0.058 0.412] | LRs: [0.001, 0.0003, 0.0003, 0.0003, 0.001]
Epoch   2 | Ens RMSE: 54.3092 | Weights: [0.275 0.096 0.158 0.058 0.413] | LRs: [0.001, 0.0003, 0.0003, 0.0003, 0.001]
Epoch   3 | Ens RMSE: 53.8847 | Weights: [0.274 0.096 0.158 0.058 0.415] | LRs: [0.001, 0.0003, 0.0003, 0.0003, 0.001]
Epoch   4 | Ens RMSE: 50.0042 | Weights: [0.272 0.096 0.158 0.0

MANTIS - DORA Predictions agent system, designed to use GenAI for metaparameter optimisation during training 